# Coinbase → Polymarket Lag Analysis

**Question:** When BTC moves sharply on Coinbase, how long does Polymarket take to reprice
its 5-minute up/down markets? And is the stale price large enough to trade after fees?

**Data required (both collected by C++ harvesters on the same AWS instance):**
- `btc_YYYYMMDD_HHMM.parquet` — from `coinbase_harvester` (BtcTick journals → hourly_flush → Parquet)
- `polymarket_YYYYMMDD_HHMM.parquet` — from `polymarket_harvester` (Tick journals → Parquet)
- `market_metadata.csv` — market questions + token IDs

Both harvesters use the same local system clock, so timestamps are directly joinable.

**Output:** lag distribution, edge per event, expected daily P&L estimate

In [ ]:
import duckdb
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths (adjust to match your S3 sync or local data dir) ───────────────────
DATA_DIR     = Path('/opt/polymarket/data')
METADATA_CSV = DATA_DIR / 'market_metadata.csv'
POLY_PARQUET = str(DATA_DIR / 'parquet' / 'polymarket_*.parquet')
BTC_PARQUET  = str(DATA_DIR / 'parquet' / 'btc_*.parquet')

# ── Strategy params ───────────────────────────────────────────────────────────
BTC_MOVE_THRESHOLD  = 0.003   # 0.3% BTC move to flag an event
MOVE_WINDOW_MS      = 500     # look for move within this many ms
MIN_EDGE_AFTER_FEES = 0.03    # minimum net edge (3%) to count as tradeable
FEE_RATE            = 0.25    # Polymarket dynamic fee rate constant
FEE_EXPONENT        = 2       # fee exponent
BTC_VOL_ANNUAL      = 0.80    # BTC annualized vol assumption

print('Paths set.')
print(f'Both datasets from C++ harvesters (same local clock, joinable by ts_ms)')
print(f'  Polymarket: {POLY_PARQUET}')
print(f'  Coinbase:    {BTC_PARQUET}')

## 1. Load Coinbase BTC Data (from C++ harvester Parquet files)

In [ ]:
con = duckdb.connect()

# Load Coinbase BTC data from Parquet (written by coinbase_harvester → btc_to_parquet.py)
btc = con.execute(f"""
    SELECT
        CAST(epoch_ms(timestamp) AS BIGINT) AS ts_ms,
        bid, ask, mid
    FROM read_parquet('{BTC_PARQUET}')
    ORDER BY ts_ms
""").df()

print(f"Coinbase rows  : {len(btc):,}")
print(f"Time range    : {pd.to_datetime(btc.ts_ms.min(), unit='ms', utc=True)} "
      f"→ {pd.to_datetime(btc.ts_ms.max(), unit='ms', utc=True)}")
print(f"BTC mid range : ${btc.mid.min():,.0f} – ${btc.mid.max():,.0f}")
print(f"Avg spread    : ${(btc.ask - btc.bid).mean():.2f}")
btc.head(3)

## 2. Load Polymarket Tick Data (up/down markets only)

In [ ]:
con = duckdb.connect()

# Load metadata and filter to 'up or down' markets
meta = pd.read_csv(METADATA_CSV)
updown_meta = meta[meta['question'].str.lower().str.contains('up or down', na=False)].copy()
print(f"Up/down markets in metadata: {len(updown_meta)}")
print(f"Unique conditions: {updown_meta['condition_id'].nunique()}")

# Register for DuckDB filtering
updown_tokens = updown_meta['asset_id'].dropna().astype(str).tolist()
con.register('meta_df', updown_meta)

# Load tick data — only price_change events for up/down tokens
token_list = "', '".join(updown_tokens)
ticks = con.execute(f"""
    SELECT
        CAST(epoch_ms(timestamp) AS BIGINT) AS ts_ms,
        asset_id,
        best_bid,
        best_ask,
        (best_bid + best_ask) / 2.0 AS mid
    FROM read_parquet('{PARQUET_GLOB}')
    WHERE event_type = 'PRICE_CHANGE'
      AND best_bid > 0
      AND best_ask > 0
      AND asset_id IN ('{token_list}')
    ORDER BY ts_ms
""").df()

print(f"Polymarket ticks loaded: {len(ticks):,}")
print(f"Unique tokens          : {ticks.asset_id.nunique()}")
ticks.head(3)

## 3. Detect BTC Move Events on Coinbase

A "move event" = BTC mid changes by ≥ `BTC_MOVE_THRESHOLD` within `MOVE_WINDOW_MS`.
These are the moments where Polymarket should reprice but might not have yet.

In [ ]:
# Rolling % change over MOVE_WINDOW_MS
# Merge-asof against a version of itself shifted by MOVE_WINDOW_MS
btc_shifted = btc[['ts_ms', 'mid']].copy()
btc_shifted['ts_ms_window'] = btc_shifted['ts_ms'] + MOVE_WINDOW_MS

# For each row, find the mid price MOVE_WINDOW_MS earlier
btc_ref = pd.merge_asof(
    btc[['ts_ms', 'mid']].rename(columns={'mid': 'mid_now'}),
    btc[['ts_ms', 'mid']].rename(columns={'ts_ms': 'ts_ref', 'mid': 'mid_ref'}),
    left_on='ts_ms',
    right_on='ts_ref',
    direction='backward',
    tolerance=MOVE_WINDOW_MS
)

btc_ref = btc_ref.dropna(subset=['mid_ref'])
btc_ref['pct_change'] = (btc_ref['mid_now'] - btc_ref['mid_ref']) / btc_ref['mid_ref']
btc_ref['abs_pct']    = btc_ref['pct_change'].abs()

# Flag events
events = btc_ref[btc_ref['abs_pct'] >= BTC_MOVE_THRESHOLD].copy()

# Deduplicate: keep only the first tick of each contiguous burst
events = events[events['ts_ms'].diff().fillna(9999) > MOVE_WINDOW_MS].copy()
events = events.reset_index(drop=True)

print(f"BTC move events detected: {len(events)}")
print(f"  (threshold={BTC_MOVE_THRESHOLD*100:.1f}% within {MOVE_WINDOW_MS}ms)")
print(f"  Events per day: {len(events) / (btc.ts_ms.ptp() / 86_400_000):.1f}")
print()
print("Move size distribution:")
print(events['abs_pct'].describe().apply(lambda x: f"{x:.4f}"))

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(events['abs_pct'] * 100, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('BTC move size (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('BTC Move Event Size Distribution')

# Events over time
events['hour'] = pd.to_datetime(events['ts_ms'], unit='ms', utc=True).dt.hour
events.groupby('hour').size().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Events')
axes[1].set_title('BTC Move Events by Hour of Day')
plt.tight_layout()
plt.show()

## 4. Measure Polymarket Response Lag

For each BTC move event at `T_coinbase`, find the next Polymarket price update across
any active up/down market. `lag_ms = T_polymarket - T_coinbase`.

In [ ]:
# For each event, find the first Polymarket tick that arrived AFTER the BTC move
results = []

ticks_sorted = ticks.sort_values('ts_ms').reset_index(drop=True)

for _, ev in events.iterrows():
    t0 = ev['ts_ms']
    # Find the earliest Polymarket tick after t0 (within 10 seconds)
    mask = (ticks_sorted['ts_ms'] > t0) & (ticks_sorted['ts_ms'] < t0 + 10_000)
    after = ticks_sorted[mask]
    if after.empty:
        continue

    first = after.iloc[0]
    lag_ms = first['ts_ms'] - t0

    results.append({
        'event_ts_ms'      : t0,
        'btc_pct_move'     : ev['pct_change'],
        'btc_mid_before'   : ev['mid_ref'],
        'btc_mid_after'    : ev['mid_now'],
        'first_poly_ts_ms' : first['ts_ms'],
        'lag_ms'           : lag_ms,
        'poly_mid_at_lag'  : first['mid'],
        'poly_ask_at_lag'  : first['best_ask'],
        'asset_id'         : first['asset_id'],
    })

lag_df = pd.DataFrame(results)
print(f"Events matched with Polymarket tick: {len(lag_df)} / {len(events)}")
print()
print("=== LAG DISTRIBUTION (ms) ===")
print(lag_df['lag_ms'].describe().apply(lambda x: f"{x:.1f} ms"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lag_df['lag_ms'], bins=50, color='darkorange', edgecolor='white')
axes[0].axvline(50,  color='red',   linestyle='--', label='50ms (our latency)')
axes[0].axvline(100, color='green', linestyle='--', label='100ms')
axes[0].set_xlabel('Lag (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Polymarket Response Lag Distribution')
axes[0].legend()

axes[1].scatter(lag_df['btc_pct_move'].abs() * 100, lag_df['lag_ms'],
               alpha=0.4, s=20, color='darkorange')
axes[1].set_xlabel('BTC Move Size (%)')
axes[1].set_ylabel('Lag (ms)')
axes[1].set_title('Lag vs BTC Move Size')
plt.tight_layout()
plt.show()

p50  = lag_df['lag_ms'].quantile(0.50)
p95  = lag_df['lag_ms'].quantile(0.95)
pct_exploitable = (lag_df['lag_ms'] > 50).mean() * 100
print(f"\nP50 lag: {p50:.0f}ms  |  P95 lag: {p95:.0f}ms")
print(f"% events where lag > 50ms (exploitable with our bot): {pct_exploitable:.1f}%")

## 5. Calculate Fair Value and Edge

For each event, compute what YES probability SHOULD be after the BTC move,
and compare to the stale Polymarket ask.

The "up or down" market resolves YES if BTC closes ABOVE its price at window open.
We model this as Brownian motion:

```
P(YES) = N( ln(S_now / S_open) / (σ_per_min × √T_remaining_min) )
```

In [ ]:
# Annualized vol → per-minute vol
TRADING_MINS_PER_YEAR = 252 * 24 * 60  # crypto trades 24/7
sigma_per_min = BTC_VOL_ANNUAL / np.sqrt(TRADING_MINS_PER_YEAR)

def taker_fee(p: float) -> float:
    """Polymarket dynamic taker fee at price p."""
    return FEE_RATE * (p * (1 - p)) ** FEE_EXPONENT

def fair_yes(s_now: float, s_open: float, t_remaining_min: float) -> float:
    """
    P(BTC closes above s_open in t_remaining_min minutes given current price s_now).
    Assumes driftless Brownian motion — conservative for a short-term prediction.
    """
    if t_remaining_min <= 0:
        return 1.0 if s_now > s_open else 0.0
    d = np.log(s_now / s_open) / (sigma_per_min * np.sqrt(t_remaining_min))
    return float(norm.cdf(d))

# For each event: estimate t_remaining by parsing the market question end time
# and compute fair value using btc_mid_after as s_now
#
# For s_open: we use btc_mid_before as a proxy for the window open price
# (not exact — window may have opened earlier — but sufficient for the experiment)
# A more precise version would track BTC price at the exact window open time.

# Merge in token metadata to get market end times
token_meta = updown_meta[['asset_id', 'condition_id', 'outcome', 'end_date']].copy()
token_meta['asset_id'] = token_meta['asset_id'].astype(str)
lag_with_meta = lag_df.merge(token_meta, on='asset_id', how='left')

# Parse end time
lag_with_meta['end_ts_ms'] = pd.to_datetime(
    lag_with_meta['end_date'], utc=True, errors='coerce'
).astype(np.int64) // 1_000_000

lag_with_meta['t_remaining_min'] = (
    (lag_with_meta['end_ts_ms'] - lag_with_meta['event_ts_ms']) / 60_000
).clip(lower=0)

# Compute fair YES probability after the BTC move
lag_with_meta['fair_yes'] = lag_with_meta.apply(
    lambda r: fair_yes(r['btc_mid_after'], r['btc_mid_before'], r['t_remaining_min']),
    axis=1
)

# Edge for YES trade (buy YES when fair_yes > poly_ask + fee)
lag_with_meta['fee_yes'] = lag_with_meta['poly_ask_at_lag'].apply(taker_fee)
lag_with_meta['edge_yes'] = (
    lag_with_meta['fair_yes'] - lag_with_meta['poly_ask_at_lag'] - lag_with_meta['fee_yes']
)

# Edge for NO trade (buy NO = sell YES; fair_no = 1 - fair_yes)
lag_with_meta['fair_no'] = 1.0 - lag_with_meta['fair_yes']
poly_no_ask = 1.0 - lag_with_meta['poly_ask_at_lag']  # proxy: no_ask ≈ 1 - yes_bid ≈ 1 - yes_ask
lag_with_meta['fee_no']   = poly_no_ask.apply(taker_fee)
lag_with_meta['edge_no']  = lag_with_meta['fair_no'] - poly_no_ask - lag_with_meta['fee_no']

# Best edge for each event
lag_with_meta['best_edge'] = lag_with_meta[['edge_yes', 'edge_no']].max(axis=1)
lag_with_meta['best_side'] = np.where(
    lag_with_meta['edge_yes'] > lag_with_meta['edge_no'], 'YES', 'NO'
)

tradeable = lag_with_meta[
    (lag_with_meta['best_edge'] >= MIN_EDGE_AFTER_FEES) &
    (lag_with_meta['lag_ms'] > 50)  # we need lag > our own latency
]

print(f"Events with edge >= {MIN_EDGE_AFTER_FEES*100:.0f}% after fees AND lag > 50ms: "
      f"{len(tradeable)} / {len(lag_with_meta)}")
print()
print("=== EDGE DISTRIBUTION (tradeable events only) ===")
print(tradeable['best_edge'].describe().apply(lambda x: f"{x:.4f}"))

## 6. Expected Daily P&L Estimate

In [ ]:
data_days = (btc['ts_ms'].max() - btc['ts_ms'].min()) / 86_400_000
POSITION_SIZE_USDC = 500.0  # conservative — limited by Polymarket liquidity

events_per_day        = len(tradeable) / data_days
avg_edge              = tradeable['best_edge'].mean()
est_pnl_per_day       = events_per_day * avg_edge * POSITION_SIZE_USDC
est_pnl_per_month     = est_pnl_per_day * 30

print("=" * 55)
print("COINBASE LAG STRATEGY — EXPECTED RETURN ESTIMATE")
print("=" * 55)
print(f"Data period           : {data_days:.1f} days")
print(f"Total BTC move events : {len(events)}")
print(f"Tradeable events      : {len(tradeable)}")
print(f"Tradeable / day       : {events_per_day:.1f}")
print(f"Avg edge after fees   : {avg_edge*100:.2f}%")
print(f"Position size         : ${POSITION_SIZE_USDC:.0f} USDC")
print(f"Est P&L / day         : ${est_pnl_per_day:.2f}")
print(f"Est P&L / month       : ${est_pnl_per_month:.2f}")
print("=" * 55)
print()
print("NOTE: These are UPPER BOUND estimates.")
print("Actual returns depend on:")
print("  - Polymarket ask liquidity at moment of trade")
print("  - Bot order actually filling before MM cancels")
print("  - S_open proxy accuracy (btc_mid_before vs true window open)")

# Breakdown by direction
print()
print("Tradeable events by side:")
print(tradeable['best_side'].value_counts().to_string())

# Edge vs lag scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    tradeable['lag_ms'], tradeable['best_edge'] * 100,
    alpha=0.5, s=25, c='green'
)
axes[0].set_xlabel('Lag (ms)')
axes[0].set_ylabel('Edge after fees (%)')
axes[0].set_title('Tradeable Events: Edge vs Lag')

axes[1].hist(tradeable['best_edge'] * 100, bins=30, color='green', edgecolor='white')
axes[1].set_xlabel('Edge after fees (%)')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Distribution (tradeable events)')

plt.tight_layout()
plt.show()

## 7. Decision Gate

Use this cell to decide whether to build the strategy into the bot.

In [ ]:
# Decision criteria
CRITERIA = {
    'Min tradeable events / day' : (events_per_day,          2.0,   '>='),
    'Avg edge after fees'        : (avg_edge,                 0.03,  '>='),
    'P50 lag > our latency (ms)' : (lag_df.lag_ms.median(),  50.0,  '>='),
    'Est monthly P&L'            : (est_pnl_per_month,        500.0, '>='),
}

print("=" * 60)
print("GO / NO-GO DECISION GATE")
print("=" * 60)
all_pass = True
for criterion, (actual, threshold, op) in CRITERIA.items():
    if op == '>=':
        passed = actual >= threshold
    result = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f"  {'✓' if passed else '✗'}  {criterion:<35} "
          f"actual={actual:>8.2f}  threshold={threshold:>8.2f}  [{result}]")

print("=" * 60)
if all_pass:
    print("VERDICT: GO — build CoinbaseFeed + LatencyArbEngine into the bot")
else:
    print("VERDICT: NO-GO — edge is insufficient. Consider:")
    print("  1. Lower BTC_MOVE_THRESHOLD (catch more events)")
    print("  2. Increase POSITION_SIZE_USDC if liquidity allows")
    print("  3. Move to sports sportsbook lag strategy instead")